# Optimized Pythia-1B: full INT4 KV-cache

Этот notebook использует проверенную реализацию из `backend/model.py`: Ocean attention, sliding-window/local attention, semantic routing и `INT4RoutedKVCache`. Здесь K/V каждого токена сохраняются в полном INT4 cache.

Для исключения расхождений определения классов они импортируются из канонического backend-модуля.


In [1]:
import gc
import json
import math
import time
from pathlib import Path
from pprint import pprint
from pprint import pprint

import torch
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint as activation_checkpoint
from datasets import load_dataset

SOURCE_MODULE = Path("/home/froschin/work/llm/backend/model.py")
if not SOURCE_MODULE.exists():
    raise FileNotFoundError(SOURCE_MODULE)

torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float16 if DEVICE.type == "cuda" else torch.float32


def clear_gpu_cache():
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


def synchronize():
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()


print({"device": str(DEVICE), "dtype": str(DTYPE), "source": str(SOURCE_MODULE)})


/home/froschin/work/llm/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'device': 'cuda', 'dtype': 'torch.float16', 'source': '/home/froschin/work/llm/backend/model.py'}


## 1. Импорт проверенной реализации модели

Исполняются только архитектурные ячейки исходного notebook: ручная Pythia, RoPE, BoundedOcean attention, chunked path, INT4 full KV-cache и routed model. Benchmark-ячеек и альтернативных моделей исходного notebook здесь нет.

In [2]:
import sys
BACKEND_DIR = SOURCE_MODULE.parent
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))
from model import (
    INT4FullKVCache,
    INT4RoutedKVCache,
    OceanAttention,
    OceanINT4PythiaForCausalLM,
    PythiaConfig,
    PythiaForCausalLM,
    ROUTING_CONFIG,
)
BoundedOceanAttention = OceanAttention
INT4_ROUTED_CONFIG = dict(ROUTING_CONFIG)
class INT4RoutedPythiaForCausalLM(OceanINT4PythiaForCausalLM):
    def __init__(self, config, **routing):
        super().__init__(config, routing=routing or INT4_ROUTED_CONFIG)
required = [
    "PythiaForCausalLM",
    "BoundedOceanAttention",
    "INT4FullKVCache",
    "INT4RoutedKVCache",
    "INT4RoutedPythiaForCausalLM",
]
print("verified classes:", required)
print("routing configuration:", INT4_ROUTED_CONFIG)


device: cuda
dtype: torch.bfloat16
torch: 2.14.0+cu126
PythiaConfig(vocab_size=50304, hidden_size=2048, intermediate_size=8192, num_hidden_layers=16, num_attention_heads=8, max_position_embeddings=2048, rotary_pct=0.25, rotary_emb_base=10000.0, rope_scaling_factor=1.0, rope_native_context=2048, layer_norm_eps=1e-05, hidden_act='gelu', use_parallel_residual=True, use_cache=True, attention_bias=True)
head_dim: 256
rotary_ndims: 64
verified classes: ['PythiaForCausalLM', 'BoundedOceanAttention', 'INT4FullKVCache', 'INT4RoutedKVCache', 'INT4RoutedPythiaForCausalLM']
routing configuration: {'block_size': 256, 'route_blocks': 16, 'beam_width': 32, 'summary_parts': 4, 'global_blocks': 1, 'local_blocks': 2, 'local_window': 256, 'route_refresh_interval': 64}


## 2. Загрузка весов в оптимизированную модель

Создаётся только одна модель `INT4RoutedPythiaForCausalLM`. `AutoModelForCausalLM` не используется.

In [3]:
from huggingface_hub import snapshot_download
from safetensors.torch import load_file as load_safetensors
from transformers import AutoTokenizer

MODEL_ID = "EleutherAI/pythia-1b"
MODEL_DIR = Path(
    snapshot_download(
        repo_id=MODEL_ID,
        allow_patterns=[
            "config.json",
            "tokenizer*",
            "*.json",
            "*.safetensors",
            "*.bin",
        ],
    )
)
tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR), use_fast=True)


def load_weight_file(path):
    if path.suffix == ".safetensors":
        return load_safetensors(str(path), device="cpu")
    try:
        return torch.load(path, map_location="cpu", weights_only=True)
    except TypeError:
        return torch.load(path, map_location="cpu")


def load_official_weights(model, root):
    root = Path(root)
    index = next(
        (
            p
            for p in (
                root / "model.safetensors.index.json",
                root / "pytorch_model.bin.index.json",
            )
            if p.exists()
        ),
        None,
    )
    if index is not None:
        state = {}
        info = json.loads(index.read_text())
        for name in sorted(set(info["weight_map"].values())):
            state.update(load_weight_file(root / name))
    else:
        path = next(
            (
                p
                for p in (root / "model.safetensors", root / "pytorch_model.bin")
                if p.exists()
            ),
            None,
        )
        state = load_weight_file(path)
    expected = set(model.state_dict())
    filtered = {k: v for k, v in state.items() if k in expected}
    missing, unexpected = model.load_state_dict(filtered, strict=False)
    if missing or unexpected:
        raise RuntimeError({"missing": missing[:20], "unexpected": unexpected[:20]})
    print("ignored auxiliary checkpoint keys:", len(set(state) - expected))
    return model


model = INT4RoutedPythiaForCausalLM(PythiaConfig(), **INT4_ROUTED_CONFIG)
model = load_official_weights(model, MODEL_DIR).to(device=DEVICE, dtype=DTYPE).eval()
print(
    {
        "model": "INT4RoutedPythiaForCausalLM",
        "parameters": sum(p.numel() for p in model.parameters()),
        "cache": "full exact INT4",
        "config": INT4_ROUTED_CONFIG,
    }
)


Fetching 6 files: 100%|██████████| 6/6 [00:00<00:00, 1319.79it/s]


ignored auxiliary checkpoint keys: 48
{'model': 'INT4RoutedPythiaForCausalLM', 'parameters': 1011781632, 'cache': 'full exact INT4', 'config': {'block_size': 256, 'route_blocks': 16, 'beam_width': 32, 'summary_parts': 4, 'global_blocks': 1, 'local_blocks': 2, 'local_window': 256, 'route_refresh_interval': 64}}


## 3. Full INT4 speed benchmark

Каждый токен записывается в полный packed INT4 K/V cache. Routing влияет только на чтение: attention получает local window и выбранные semantic blocks.

In [4]:
def repeated_prompt(token_ids, length):
    source = token_ids.flatten().to(DEVICE)
    return source.repeat(math.ceil(length / source.numel()))[:length]


def estimate_full_int4_kv_gib(config, context_length):
    elements = (
        2
        * config.num_hidden_layers
        * config.num_attention_heads
        * context_length
        * config.head_dim
    )
    packed = math.ceil(elements / 2)
    scales = (
        2 * config.num_hidden_layers * config.num_attention_heads * context_length * 2
    )
    return (packed + scales) / 2 ** 30


@torch.inference_mode()
def benchmark_full_int4(model, token_ids, prompt_length, new_tokens=16, chunk_size=256):
    ids = repeated_prompt(token_ids, prompt_length)
    caches = model.new_bounded_cache(prompt_length + new_tokens)
    try:
        if DEVICE.type == "cuda":
            torch.cuda.reset_peak_memory_stats()
        synchronize()
        start = time.perf_counter()
        logits = None
        for chunk_start in range(0, prompt_length, chunk_size):
            chunk = ids[chunk_start : min(chunk_start + chunk_size, prompt_length)]
            logits = model.forward_bounded_chunk(chunk, caches, chunk_start)
            processed = chunk_start + chunk.numel()
            if processed % 100000 < chunk_size or processed == prompt_length:
                print(f"full INT4 prefill progress: {processed:,}/{prompt_length:,}")
        synchronize()
        prefill = time.perf_counter() - start
        next_token = logits[:, -1:].argmax(-1)
        synchronize()
        start = time.perf_counter()
        for position in range(prompt_length, prompt_length + new_tokens):
            logits = model.forward_bounded_token(next_token[:, 0], caches, position)
            next_token = logits[:, -1:].argmax(-1)
        synchronize()
        decode = time.perf_counter() - start
        out = {
            "cache": "full_exact_int4_kv_routed",
            "prompt_length": prompt_length,
            "chunk_size": chunk_size,
            "new_tokens": new_tokens,
            "prefill_seconds": prefill,
            "prefill_tokens_per_second": prompt_length / prefill,
            "decode_seconds": decode,
            "decode_tokens_per_second": new_tokens / decode,
            "total_seconds": prefill + decode,
            "estimated_full_int4_kv_gib": estimate_full_int4_kv_gib(
                model.config, prompt_length + new_tokens
            ),
            "status": "ok",
        }
        if DEVICE.type == "cuda":
            out.update(
                {
                    "peak_cuda_allocated_gib": torch.cuda.max_memory_allocated()
                    / 2 ** 30,
                    "peak_cuda_reserved_gib": torch.cuda.max_memory_reserved()
                    / 2 ** 30,
                }
            )
        return out
    finally:
        del caches, ids


def run_speed_sweep(
    token_ids,
    prompt_lengths=(2048, 14000, 32000, 100000, 1000000),
    new_tokens=16,
    chunk_size=256,
    memory_guard_gib=26.0,
    force_large_cache=False,
):
    results = []
    for length in prompt_lengths:
        estimate = estimate_full_int4_kv_gib(model.config, length + new_tokens)
        clear_gpu_cache()
        if (
            not force_large_cache
            and memory_guard_gib is not None
            and estimate > memory_guard_gib
        ):
            row = {
                "cache": "full_exact_int4_kv_routed",
                "prompt_length": length,
                "estimated_full_int4_kv_gib": estimate,
                "status": "skipped_memory_guard",
            }
        else:
            try:
                row = benchmark_full_int4(
                    model, token_ids, length, new_tokens, chunk_size
                )
            except RuntimeError as exc:
                if "out of memory" not in str(exc).lower():
                    raise
                row = {
                    "cache": "full_exact_int4_kv_routed",
                    "prompt_length": length,
                    "estimated_full_int4_kv_gib": estimate,
                    "status": "cuda_oom",
                    "error": str(exc).split("\n")[0],
                }
        print(row)
        results.append(row)
    return results


RUN_SPEED = False
if RUN_SPEED:
    source_ids = torch.tensor(
        tokenizer("To be, or not to be. " * 10000, add_special_tokens=False).input_ids,
        dtype=torch.long,
    )
    speed_results = run_speed_sweep(source_ids)


## 4. Continuous PPL с полным INT4 cache

In [5]:
@torch.inference_mode()
def evaluate_full_int4_ppl(model, token_ids, context_length=2048, chunk_size=256):
    ids = token_ids.flatten()[:context_length].to(DEVICE)
    caches = model.new_bounded_cache(ids.numel())
    total_nll = 0.0
    total = 0
    try:
        synchronize()
        start = time.perf_counter()
        logits = None
        for pos in range(0, ids.numel(), chunk_size):
            end = min(pos + chunk_size, ids.numel())
            logits = model.forward_bounded_chunk(ids[pos:end], caches, pos)
            target = ids[pos + 1 : min(end + 1, ids.numel())]
            if target.numel():
                total_nll += F.cross_entropy(
                    logits[:, : target.numel(), :]
                    .float()
                    .reshape(-1, model.config.vocab_size),
                    target,
                    reduction="sum",
                ).item()
                total += target.numel()
        synchronize()
        seconds = time.perf_counter() - start
        mean = total_nll / max(total, 1)
        return {
            "cache": "full_exact_int4_kv_routed",
            "context_length": int(ids.numel()),
            "tokens": total,
            "mean_nll": mean,
            "perplexity": math.exp(mean),
            "seconds": seconds,
            "tokens_per_second": total / max(seconds, 1e-9),
        }
    finally:
        del caches


RUN_PPL = False
if RUN_PPL:
    pg19 = load_dataset("emozilla/pg19", split="train", streaming=True)
    record = next(iter(pg19))
    pg19_ids = torch.tensor(
        tokenizer(record["text"], add_special_tokens=False).input_ids, dtype=torch.long
    )
    for context in (2048, 8192, 16384):
        clear_gpu_cache()
        print(evaluate_full_int4_ppl(model, pg19_ids, context))


## 5. Continued pretraining на больших датасетах

Для обучения используется тот же объект `model`, но перед стартом включается `TrainableRoutedAttention`: блоки выбираются по detached summary, а attention по выбранным K/V дифференцируем. INT4 cache применяется только на inference и не включается в autograd-граф.

In [6]:
class TrainableRoutedAttention(BoundedOceanAttention):
    # Индексы блоков выбираются дискретно, но attention по выбранным K/V
    # остаётся дифференцируемым и участвует в backpropagation.
    def __init__(self, config, block_size=256, local_window=256, memory_slots=16, summary_parts=4, global_blocks=1, local_blocks=2, route_refresh_interval=64):
        super().__init__(config, block_size=block_size, local_window=local_window, memory_slots=memory_slots)
        if block_size % summary_parts != 0:
            raise ValueError('block_size должен делиться на summary_parts')
        self.summary_parts = int(summary_parts)
        self.part_size = int(block_size // summary_parts)
        self.global_blocks = int(global_blocks)
        self.local_blocks = int(local_blocks)
        self.route_refresh_interval = max(1, int(route_refresh_interval))

    @torch.no_grad()
    def _select_training_blocks(self, query, key, routeable_blocks):
        if routeable_blocks <= 0 or self.memory_slots <= 0:
            return torch.empty(1, self.num_attention_heads, 0, device=key.device, dtype=torch.long)
        route_count = min(int(self.memory_slots), int(routeable_blocks))
        local_count = min(self.local_blocks, route_count)
        global_count = min(self.global_blocks, max(0, route_count - local_count))
        mandatory = list(range(global_count))
        mandatory.extend(range(routeable_blocks - local_count, routeable_blocks))
        mandatory = list(dict.fromkeys(mandatory))
        mandatory_ids = torch.tensor(mandatory, device=key.device, dtype=torch.long)
        semantic_count = route_count - len(mandatory)
        if semantic_count <= 0:
            return mandatory_ids.view(1, 1, -1).expand(1, self.num_attention_heads, -1)
        usable = routeable_blocks * self.block_size
        block_key = key[:, :, :usable, :].detach()
        block_key = block_key.view(1, self.num_attention_heads, routeable_blocks, self.summary_parts, self.part_size, self.head_dim).mean(dim=4)
        query_vector = F.normalize(query.detach().mean(dim=2).float(), dim=-1).unsqueeze(2).unsqueeze(3)
        block_norm = F.normalize(block_key.float(), dim=-1)
        scores = (query_vector * block_norm).sum(dim=-1).amax(dim=-1)
        blocked = torch.zeros(routeable_blocks, device=key.device, dtype=torch.bool)
        if mandatory:
            blocked[mandatory_ids] = True
        scores = scores.masked_fill(blocked.view(1, 1, -1), float('-inf'))
        semantic = scores.topk(semantic_count, dim=-1).indices
        mandatory_part = mandatory_ids.view(1, 1, -1).expand(1, self.num_attention_heads, -1)
        return torch.cat((mandatory_part, semantic), dim=-1)

    def forward(self, hidden_states, past_key_value=None, use_cache=False):
        if past_key_value is not None or use_cache:
            raise NotImplementedError('TrainableRoutedAttention использует полный training sequence без past cache')
        batch_size, q_len, _ = hidden_states.shape
        if batch_size != 1:
            raise NotImplementedError('training routed path пока поддерживает только batch_size=1')
        qkv = self.query_key_value(hidden_states)
        qkv = qkv.view(1, q_len, self.num_attention_heads, 3 * self.head_dim).transpose(1, 2)
        query, key, value = qkv.chunk(3, dim=-1)
        positions = torch.arange(q_len, device=hidden_states.device, dtype=torch.long)
        cos, sin = self.rotary_emb(positions, hidden_states.dtype)
        query, key = apply_rotary(query, key, cos, sin, self.rotary_ndims)
        outputs = []
        for query_start in range(0, q_len, self.route_refresh_interval):
            query_end = min(query_start + self.route_refresh_interval, q_len)
            local_start = max(0, query_start - self.local_window)
            routeable_blocks = local_start // self.block_size
            # Только первый query блока выбирает маршрут: будущие токены
            # текущего блока не должны влиять на causal routing.
            route = self._select_training_blocks(query[:, :, query_start:query_start + 1, :], key, routeable_blocks)
            local_ids = torch.arange(local_start, query_end, device=hidden_states.device, dtype=torch.long)
            local_key = key.index_select(2, local_ids)
            local_value = value.index_select(2, local_ids)
            query_positions = torch.arange(query_start, query_end, device=hidden_states.device, dtype=torch.long)
            local_allowed = local_ids.view(1, 1, 1, -1) <= query_positions.view(1, 1, -1, 1)
            if route.shape[-1] > 0:
                offsets = torch.arange(self.block_size, device=hidden_states.device, dtype=torch.long)
                route_ids = (route.unsqueeze(-1) * self.block_size + offsets.view(1, 1, 1, -1)).reshape(1, self.num_attention_heads, -1)
                route_key = key.gather(2, route_ids.unsqueeze(-1).expand(-1, -1, -1, self.head_dim))
                route_value = value.gather(2, route_ids.unsqueeze(-1).expand(-1, -1, -1, self.head_dim))
                route_allowed = route_ids.unsqueeze(2) <= query_positions.view(1, 1, -1, 1)
                selected_key = torch.cat((local_key, route_key), dim=2)
                selected_value = torch.cat((local_value, route_value), dim=2)
                allowed = torch.cat((local_allowed.expand(1, self.num_attention_heads, -1, -1), route_allowed), dim=-1)
            else:
                selected_key = local_key
                selected_value = local_value
                allowed = local_allowed.expand(1, self.num_attention_heads, -1, -1)
            output = F.scaled_dot_product_attention(
                query[:, :, query_start:query_end, :],
                selected_key, selected_value,
                attn_mask=allowed, dropout_p=0.0, is_causal=False,
            )
            outputs.append(output)
        attention_output = torch.cat(outputs, dim=2).transpose(1, 2).contiguous().view(1, q_len, -1)
        return self.dense(attention_output), None


def forward_trainable_routed(model, input_ids, gradient_checkpointing=False):
    hidden_states = model.gpt_neox.embed_in(input_ids)
    for layer in model.gpt_neox.layers:
        def layer_forward(states, current_layer=layer):
            output, _ = current_layer(states, past_key_value=None, use_cache=False)
            return output
        if gradient_checkpointing and model.training:
            hidden_states = activation_checkpoint(
                layer_forward, hidden_states, use_reentrant=False
            )
        else:
            hidden_states = layer_forward(hidden_states)
    hidden_states = model.gpt_neox.final_layer_norm(hidden_states)
    return model.embed_out(hidden_states), None


def enable_trainable_routing(model):
    if getattr(model, '_trainable_routing_enabled', False):
        return model
    for layer in model.gpt_neox.layers:
        old_attention = layer.attention
        new_attention = TrainableRoutedAttention(
            model.config,
            block_size=INT4_ROUTED_CONFIG['block_size'],
            local_window=INT4_ROUTED_CONFIG['local_window'],
            memory_slots=INT4_ROUTED_CONFIG['route_blocks'],
            summary_parts=INT4_ROUTED_CONFIG['summary_parts'],
            global_blocks=INT4_ROUTED_CONFIG['global_blocks'],
            local_blocks=INT4_ROUTED_CONFIG['local_blocks'],
            route_refresh_interval=INT4_ROUTED_CONFIG['route_refresh_interval'],
        )
        new_attention.load_state_dict(old_attention.state_dict(), strict=True)
        reference_parameter = next(old_attention.parameters())
        new_attention.to(device=reference_parameter.device, dtype=reference_parameter.dtype)
        layer.attention = new_attention
    model._trainable_routing_enabled = True
    print('trainable routed attention enabled')
    return model


In [7]:
from transformers import get_cosine_schedule_with_warmup

DATASET_NAME = "pg19"
DATASET_SPLIT = "train"
DATASET_CONFIG = None
TRAIN_BLOCK_SIZE = 2048
TRAIN_MAX_STEPS = 5000
GRAD_ACCUM = 8
LEARNING_RATE = 1e-6
MAX_TRAIN_TOKENS = TRAIN_MAX_STEPS * TRAIN_BLOCK_SIZE
TRAIN_FP32_MASTER = True
TRAIN_WITH_ROUTED_ATTENTION = True
SAVE_OPTIMIZER_STATE = True
CHECKPOINT_DIR = Path("/home/froschin/work/llm/checkpoints/pythia-ocean-int4")


def text_stream(name=DATASET_NAME, split=DATASET_SPLIT, config_name=DATASET_CONFIG, block_size=None, max_tokens=None):
    block_size = TRAIN_BLOCK_SIZE if block_size is None else int(block_size)
    max_tokens = MAX_TRAIN_TOKENS if max_tokens is None else max_tokens
    if name == "pg19":
        ds = load_dataset("emozilla/pg19", split=split, streaming=True)
    elif name == "dolma":
        ds = load_dataset("allenai/dolma", split=split, streaming=True)
    elif name == "fineweb":
        ds = load_dataset(
            "HuggingFaceFW/fineweb",
            config_name or "sample-10BT",
            split=split,
            streaming=True,
        )
    else:
        raise ValueError(name)
    buf = []
    seen = 0
    for row in ds:
        text = row.get("text", row.get("content", row.get("document", "")))
        ids = tokenizer(text, add_special_tokens=False).input_ids
        if max_tokens is not None:
            ids = ids[: max(0, max_tokens - seen)]
        buf.extend(ids)
        seen += len(ids)
        while len(buf) >= block_size:
            yield torch.tensor(buf[:block_size], dtype=torch.long)
            del buf[:block_size]
        if max_tokens is not None and seen >= max_tokens:
            break


LAST_TRAINING_STATE = {}


def train_model(model, stream, max_steps=TRAIN_MAX_STEPS, gradient_checkpointing=False, tokens_per_step=None, learning_rate=None):
    if TRAIN_WITH_ROUTED_ATTENTION:
        model = enable_trainable_routing(model)
    model = model.to(device=DEVICE)
    if not all(torch.isfinite(parameter).all() for parameter in model.parameters()):
        raise RuntimeError("В модели уже есть NaN/Inf. Перезагрузите официальные веса перед новым запуском.")
    if TRAIN_FP32_MASTER:
        model = model.float()
    model.train()
    effective_learning_rate = LEARNING_RATE if learning_rate is None else float(learning_rate)
    effective_tokens_per_step = TRAIN_BLOCK_SIZE if tokens_per_step is None else int(tokens_per_step)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=effective_learning_rate, weight_decay=0.1
    )
    optimizer_steps = math.ceil(max_steps / GRAD_ACCUM)
    scheduler = get_cosine_schedule_with_warmup(
        optimizer, min(10, optimizer_steps), optimizer_steps
    )
    parameter_dtype = next(model.parameters()).dtype
    autocast_dtype = torch.float16 if DEVICE.type == "cuda" else torch.float32
    use_scaler = (
        DEVICE.type == "cuda"
        and autocast_dtype == torch.float16
        and parameter_dtype == torch.float32
    )
    scaler = torch.cuda.amp.GradScaler(enabled=use_scaler)
    print(
        {
            "training_parameter_dtype": str(parameter_dtype),
            "grad_scaler_enabled": use_scaler,
        }
    )
    optimizer.zero_grad(set_to_none=True)
    optimizer_step = 0
    loss_ema = None
    start = time.perf_counter()
    for step in range(max_steps):
        ids = next(stream).unsqueeze(0).to(DEVICE)
        with torch.autocast(
            device_type=DEVICE.type, dtype=autocast_dtype, enabled=DEVICE.type == "cuda"
        ):
            if TRAIN_WITH_ROUTED_ATTENTION:
                logits, _ = forward_trainable_routed(
                    model, ids, gradient_checkpointing=gradient_checkpointing
                )
            else:
                logits, _ = model(ids, use_cache=False)
            loss = F.cross_entropy(
                logits[:, :-1].float().reshape(-1, model.config.vocab_size),
                ids[:, 1:].reshape(-1),
            )
            if not torch.isfinite(loss):
                raise FloatingPointError(f"NaN/Inf loss at step {step + 1}")
            scaled = loss / GRAD_ACCUM
        (scaler.scale(scaled) if use_scaler else scaled).backward()
        if (step + 1) % GRAD_ACCUM == 0:
            if use_scaler:
                scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            if use_scaler:
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()
            scheduler.step()
            optimizer_step += 1
            optimizer.zero_grad(set_to_none=True)
        loss_value = float(loss.detach())
        loss_ema = loss_value if loss_ema is None else 0.95 * loss_ema + 0.05 * loss_value
        if (step + 1) % 10 == 0:
            print(
                {
                    "step": step + 1,
                    "optimizer_step": optimizer_step,
                    "loss": float(loss),
                    "loss_ema": loss_ema,
                    "learning_rate": scheduler.get_last_lr()[0],
                    "tokens_seen": (step + 1) * effective_tokens_per_step,
                    "seconds": time.perf_counter() - start,
                }
            )
    LAST_TRAINING_STATE.clear()
    LAST_TRAINING_STATE.update(
        {
            "optimizer_state": optimizer.state_dict(),
            "scheduler_state": scheduler.state_dict(),
            "completed_steps": max_steps,
            "completed_optimizer_steps": optimizer_step,
        }
    )
    return model.eval()


RUN_TRAINING = False
if RUN_TRAINING:
    model = train_model(model, text_stream())
    CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
    torch.save(model.state_dict(), CHECKPOINT_DIR / "model_state.pt")
    checkpoint = {
        "model_state": model.state_dict(),
        "model_config": dict(model.config.__dict__),
        "routing_config": dict(INT4_ROUTED_CONFIG),
        "dataset": DATASET_NAME,
        "train_block_size": TRAIN_BLOCK_SIZE,
        "learning_rate": LEARNING_RATE,
        "completed_steps": TRAIN_MAX_STEPS,
        "train_with_routed_attention": TRAIN_WITH_ROUTED_ATTENTION,
    }
    if SAVE_OPTIMIZER_STATE:
        checkpoint.update(LAST_TRAINING_STATE)
    torch.save(checkpoint, CHECKPOINT_DIR / "training_checkpoint.pt")
    tokenizer.save_pretrained(CHECKPOINT_DIR)
    model = model.to(device=DEVICE, dtype=DTYPE).eval()
    print("saved:", CHECKPOINT_DIR)


In [8]:
LONG_CONTEXT_LENGTH = 8192
LONG_CONTEXT_MAX_STEPS = 30
LONG_CONTEXT_MAX_TOKENS = LONG_CONTEXT_LENGTH * LONG_CONTEXT_MAX_STEPS
LONG_CONTEXT_LEARNING_RATE = 5e-7
LONG_CONTEXT_CHECKPOINT_DIR = CHECKPOINT_DIR / 'long-context-8192'
LONG_CONTEXT_INIT_CHECKPOINT = CHECKPOINT_DIR / 'model_state.pt'


def save_long_context_checkpoint(model, directory, completed_steps):
    directory.mkdir(parents=True, exist_ok=True)
    torch.save(model.state_dict(), directory / 'model_state.pt')
    checkpoint = {
        'model_state': model.state_dict(),
        'model_config': dict(model.config.__dict__),
        'routing_config': dict(INT4_ROUTED_CONFIG),
        'dataset': DATASET_NAME,
        'train_context_length': LONG_CONTEXT_LENGTH,
        'train_block_size': LONG_CONTEXT_LENGTH,
        'learning_rate': LONG_CONTEXT_LEARNING_RATE,
        'completed_steps': completed_steps,
        'train_with_routed_attention': True,
        'gradient_checkpointing': True,
    }
    if SAVE_OPTIMIZER_STATE:
        checkpoint.update(LAST_TRAINING_STATE)
    torch.save(checkpoint, directory / 'training_checkpoint.pt')
    tokenizer.save_pretrained(directory)
    print('saved long-context checkpoint:', directory)


RUN_LONG_CONTEXT_TRAINING = False
if RUN_LONG_CONTEXT_TRAINING:
    if LONG_CONTEXT_INIT_CHECKPOINT.exists():
        print('loading short-context checkpoint for long-context adaptation:', LONG_CONTEXT_INIT_CHECKPOINT)
        initial_state = torch.load(LONG_CONTEXT_INIT_CHECKPOINT, map_location='cpu', weights_only=True)
        model.load_state_dict(initial_state, strict=True)
        del initial_state
        model = model.to(device=DEVICE, dtype=DTYPE).eval()
    print({
        'context_length': LONG_CONTEXT_LENGTH,
        'steps': LONG_CONTEXT_MAX_STEPS,
        'tokens': LONG_CONTEXT_MAX_TOKENS,
        'route_blocks': INT4_ROUTED_CONFIG['route_blocks'],
        'local_window': INT4_ROUTED_CONFIG['local_window'],
        'route_refresh_interval': INT4_ROUTED_CONFIG['route_refresh_interval'],
        'gradient_checkpointing': True,
    })
    long_stream = text_stream(
        block_size=LONG_CONTEXT_LENGTH,
        max_tokens=LONG_CONTEXT_MAX_TOKENS,
    )
    model = train_model(
        model,
        long_stream,
        max_steps=LONG_CONTEXT_MAX_STEPS,
        gradient_checkpointing=True,
        tokens_per_step=LONG_CONTEXT_LENGTH,
        learning_rate=LONG_CONTEXT_LEARNING_RATE,
    )
    save_long_context_checkpoint(model, LONG_CONTEXT_CHECKPOINT_DIR, LONG_CONTEXT_MAX_STEPS)
    model = model.to(device=DEVICE, dtype=DTYPE).eval()


In [9]:
LONG_CONTEXT_EVAL_DOCUMENTS = 4
LONG_CONTEXT_EVAL_LENGTHS = (2048, 8192, 16384)
LONG_CONTEXT_EVAL_CHUNK_SIZE = 256


def load_model_state_or_official(model, checkpoint_path):
    if checkpoint_path is not None and Path(checkpoint_path).exists():
        print('loading checkpoint:', checkpoint_path)
        state = torch.load(checkpoint_path, map_location='cpu', weights_only=True)
        model.load_state_dict(state, strict=True)
        del state
    else:
        print('loading official weights')
        load_official_weights(model, MODEL_DIR)
    return model.to(device=DEVICE, dtype=DTYPE).eval()


def evaluate_same_long_documents(model, documents, context_lengths, chunk_size=256):
    results = {}
    for context_length in context_lengths:
        rows = []
        total_nll = 0.0
        total_tokens = 0
        for index, ids in enumerate(documents):
            if ids.numel() < context_length:
                continue
            clear_gpu_cache()
            row = evaluate_full_int4_ppl(
                model, ids, context_length=context_length, chunk_size=chunk_size
            )
            row['document_index'] = index
            rows.append(row)
            total_nll += row['mean_nll'] * row['tokens']
            total_tokens += row['tokens']
        if not rows:
            results[context_length] = {'status': 'no_document_long_enough'}
            continue
        mean_nll = total_nll / total_tokens
        results[context_length] = {
            'context_length': context_length,
            'documents': len(rows),
            'tokens': total_tokens,
            'mean_nll': mean_nll,
            'perplexity': math.exp(mean_nll),
            'per_document': rows,
        }
    return results


def run_long_context_before_after_benchmark():
    documents = load_pg19_validation_ids(
        max_documents=LONG_CONTEXT_EVAL_DOCUMENTS,
        max_tokens_per_document=max(LONG_CONTEXT_EVAL_LENGTHS),
    )
    short_checkpoint = CHECKPOINT_DIR / 'model_state.pt'
    long_checkpoint = LONG_CONTEXT_CHECKPOINT_DIR / 'model_state.pt'
    if not long_checkpoint.exists():
        raise FileNotFoundError(
            f'Сначала выполните long-context training: {long_checkpoint}'
        )
    print({
        'documents': len(documents),
        'context_lengths': LONG_CONTEXT_EVAL_LENGTHS,
        'same_documents_and_order': True,
        'baseline': str(short_checkpoint) if short_checkpoint.exists() else 'official_weights',
        'adapted': str(long_checkpoint),
    })
    load_model_state_or_official(model, short_checkpoint if short_checkpoint.exists() else None)
    before = evaluate_same_long_documents(
        model, documents, LONG_CONTEXT_EVAL_LENGTHS, LONG_CONTEXT_EVAL_CHUNK_SIZE
    )
    clear_gpu_cache()
    load_model_state_or_official(model, long_checkpoint)
    after = evaluate_same_long_documents(
        model, documents, LONG_CONTEXT_EVAL_LENGTHS, LONG_CONTEXT_EVAL_CHUNK_SIZE
    )
    comparison = {}
    for context_length in LONG_CONTEXT_EVAL_LENGTHS:
        before_row = before[context_length]
        after_row = after[context_length]
        if 'perplexity' not in before_row or 'perplexity' not in after_row:
            comparison[context_length] = {'status': 'missing_result'}
            continue
        comparison[context_length] = {
            'before_ppl': before_row['perplexity'],
            'after_ppl': after_row['perplexity'],
            'ppl_delta': after_row['perplexity'] - before_row['perplexity'],
            'ppl_relative_percent': 100.0 * (after_row['perplexity'] / before_row['perplexity'] - 1.0),
            'before_mean_nll': before_row['mean_nll'],
            'after_mean_nll': after_row['mean_nll'],
        }
    result = {
        'protocol': {
            'documents': len(documents),
            'context_lengths': LONG_CONTEXT_EVAL_LENGTHS,
            'same_documents_and_order': True,
            'cache': 'full_exact_int4_kv_routed',
        },
        'before_long_context_adaptation': before,
        'after_long_context_adaptation': after,
        'comparison': comparison,
    }
    print('--- long-context INT4 routed: before vs after adaptation ---')
    pprint(comparison)
    return result


RUN_LONG_CONTEXT_BEFORE_AFTER_BENCHMARK = False
if RUN_LONG_CONTEXT_BEFORE_AFTER_BENCHMARK:
    long_context_before_after_results = run_long_context_before_after_benchmark()


## 6. Независимый validation PPL на нескольких документах PG-19

Validation всегда загружается из `validation`; train-документы и переменная `pg19_ids` не переиспользуются.

In [12]:
VALIDATION_DOCUMENTS = 64


def load_pg19_validation_ids(max_documents=VALIDATION_DOCUMENTS, max_tokens_per_document=2048):
    dataset = load_dataset("emozilla/pg19", split="validation", streaming=True)
    documents = []
    for record in dataset:
        ids = tokenizer(record["text"], add_special_tokens=False).input_ids
        if len(ids) >= 2:
            documents.append(torch.tensor(ids[:max_tokens_per_document], dtype=torch.long))
        if len(documents) >= max_documents:
            break
    if not documents:
        raise RuntimeError("PG-19 validation не вернул документов")
    return documents

@torch.inference_mode()
def evaluate_pg19_validation(model, max_documents=VALIDATION_DOCUMENTS, context_length=2048, chunk_size=256, documents=None):
    if documents is None:
        documents = load_pg19_validation_ids(max_documents, context_length)
    rows = []
    total_nll = 0.0
    total_tokens = 0
    for index, ids in enumerate(documents):
        clear_gpu_cache()
        row = evaluate_full_int4_ppl(model, ids, context_length=min(context_length, ids.numel()), chunk_size=chunk_size)
        row["document_index"] = index
        rows.append(row)
        total_nll += row["mean_nll"] * row["tokens"]
        total_tokens += row["tokens"]
    mean_nll = total_nll / max(total_tokens, 1)
    return {
        "split": "validation",
        "documents": len(rows),
        "tokens": total_tokens,
        "mean_nll": mean_nll,
        "perplexity": math.exp(mean_nll),
        "per_document": rows,
    }

RUN_VALIDATION_PPL = False
if RUN_VALIDATION_PPL:
    pprint(evaluate_pg19_validation(model, max_documents=VALIDATION_DOCUMENTS))


{'documents': 50,
 'mean_nll': 2.507358616641165,
 'per_document': [{'cache': 'full_exact_int4_kv_routed',
                   'context_length': 2048,
                   'document_index': 0,
                   'mean_nll': 2.908518498154693,
                   'perplexity': 18.32962307131098,
                   'seconds': 0.9781717772129923,
                   'tokens': 2047,
                   'tokens_per_second': 2092.6794737753667},
                  {'cache': 'full_exact_int4_kv_routed',
                   'context_length': 2048,
                   'document_index': 1,
                   'mean_nll': 3.065453063935027,
                   'perplexity': 21.44417536910009,
                   'seconds': 0.6283435199875385,
                   'tokens': 2047,
                   'tokens_per_second': 3257.7721180932954},
                  {'cache': 'full_exact_int4_kv_routed',
                   'context_length': 2048,
                   'document_index': 2,
                   'mean_nll': 2.6

## 6. Сравнение до и после continued pretraining

Сначала в тот же объект загружаются исходные официальные веса, затем checkpoint после обучения. Используются один и тот же PG-19-фрагмент, одинаковый full INT4 routed path и одинаковый speed protocol.

In [13]:
def run_post_training_comparison(context_length=2048, new_tokens=16, chunk_size=256):
    checkpoint_path = CHECKPOINT_DIR / "model_state.pt"
    if not checkpoint_path.exists():
        raise FileNotFoundError(f"Сначала выполните обучение: {checkpoint_path}")
    clear_gpu_cache()
    eval_documents = load_pg19_validation_ids(max_documents=VALIDATION_DOCUMENTS, max_tokens_per_document=context_length)
    eval_ids = eval_documents[0]
    print("loading original official weights")
    load_official_weights(model, MODEL_DIR)
    model.to(device=DEVICE, dtype=DTYPE).eval()
    before_ppl = evaluate_pg19_validation(model, max_documents=VALIDATION_DOCUMENTS, context_length=context_length, chunk_size=chunk_size, documents=eval_documents)
    before_speed = benchmark_full_int4(
        model, eval_ids, context_length, new_tokens, chunk_size
    )
    clear_gpu_cache()
    print("loading trained checkpoint:", checkpoint_path)
    trained_state = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
    model.load_state_dict(trained_state, strict=True)
    del trained_state
    model.to(device=DEVICE, dtype=DTYPE).eval()
    after_ppl = evaluate_pg19_validation(model, max_documents=VALIDATION_DOCUMENTS, context_length=context_length, chunk_size=chunk_size, documents=eval_documents)
    after_speed = benchmark_full_int4(
        model, eval_ids, context_length, new_tokens, chunk_size
    )
    result = {
        "before_training": {"ppl": before_ppl, "speed": before_speed},
        "after_training": {"ppl": after_ppl, "speed": after_speed},
        "ppl_delta": after_ppl["perplexity"] - before_ppl["perplexity"],
        "decode_tok_s_delta": after_speed["decode_tokens_per_second"]
        - before_speed["decode_tokens_per_second"],
    }
    print("--- post-training comparison ---")
    pprint(result)
    return result


RUN_POST_TRAINING_EVAL = False
if RUN_POST_TRAINING_EVAL:
    post_training_results = run_post_training_comparison()


loading original official weights
ignored auxiliary checkpoint keys: 48
full INT4 prefill progress: 2,048/2,048
loading trained checkpoint: /home/froschin/work/llm/checkpoints/pythia-ocean-int4/model_state.pt
full INT4 prefill progress: 2,048/2,048
--- post-training comparison ---
{'after_training': {'ppl': {'documents': 50,
                            'mean_nll': 2.6086049514569245,
                            'per_document': [{'cache': 'full_exact_int4_kv_routed',
                                              'context_length': 2048,
                                              'document_index': 0,
                                              'mean_nll': 3.0273698397712354,
                                              'perplexity': 20.64286707773124,
                                              'seconds': 0.6753265738952905,
                                              'tokens': 2047,
                                              'tokens_per_second': 3031.1260938435803},
        

## Ограничения

`full_exact_int4_kv_routed` хранит K/V каждого токена, поэтому память KV растёт как O(N). Для Pythia-1B это примерно 31 GiB на 1M токенов без весов модели и временных буферов; на V100 32GB такой тест обычно должен быть пропущен memory guard или закончиться OOM. Это не bounded-cache benchmark.